# 05 · Does Clay actually detect the Marshall Fire?

Notebook 03's nearest-neighbor query aimed at the Marshall Fire burn scar was inconclusive: the
neighbors it returned looked like generic suburban/grassland edge, not something visibly
"burned." But that used a single snapshot from mid-2024, ~2.5 years after the Dec 30, 2021 fire
-- no baseline, no before/after, and plenty of time for regrowth to erase any signal. This
notebook does the actual before/after comparison that entry should have been:

1. Fetch three single dates over a tight box around the actual burned area: **pre-fire**
   (fall 2021), **immediate post-fire** (winter/spring 2022), and **long-term** (summer 2024,
   matching the rest of the study).
2. Compute **dNBR** (differenced Normalized Burn Ratio) from the raw bands -- a standard,
   independent remote-sensing burn-severity index -- as ground truth for how much each chip
   actually burned.
3. Embed all three dates with Clay and measure how far each chip's embedding *moves* between
   pre-fire and each later date.
4. The real test: does embedding movement correlate with dNBR? If Clay's embeddings are
   sensitive to the burn at all, chips that burned more severely should show a bigger embedding
   shift than chips that didn't burn -- and if that correlation is strong right after the fire
   but weak by 2024, that would explain notebook 03's inconclusive result as "the signal faded,"
   not "Clay never saw it."

**Requires a GPU runtime** (T4 is plenty), but the AOI here is small (~50 chips vs. 725 for the
main study) and self-contained -- no Drive mount needed, everything happens in this one
notebook.

In [ ]:
REPO_URL = "https://github.com/ZanderHirman08/SATEMB.git"

import os

if not os.path.exists("SATEMB"):
    !git clone {REPO_URL}
%cd SATEMB
!pip install -q -r environment/requirements-colab.txt

In [ ]:
import sys

sys.path.append(os.getcwd())

import matplotlib.pyplot as plt
import numpy as np
import odc.stac
import pandas as pd
import torch
from scipy import stats

from src import clay_embed, stac_utils

assert torch.cuda.is_available(), "No GPU detected -- switch runtime type to T4 GPU and re-run"
device = "cuda"

os.makedirs("docs/figures", exist_ok=True)
catalog = stac_utils.open_catalog()
BBOX = stac_utils.MARSHALL_FIRE_BBOX
print(f"AOI: {BBOX}")

## Fetch three single dates

One specific scene per period, not a composite -- blending would blur exactly the
snow/no-snow and burned/unburned distinctions this comparison depends on. Cloud threshold is
relaxed to 20% since winter Front Range scenes have fewer clean options than the summer window
used elsewhere in this study.

**Known risk:** Denver-area got light snow the day after the fire (Dec 31, 2021), so early-2022
imagery could be snow-contaminated. The true-color sanity plots below are there specifically to
catch that before trusting anything quantitative -- check them before reading further.

In [ ]:
pre_item = stac_utils.search_single_scene(catalog, BBOX, "2021-09-15/2021-12-29", label="pre-fire")
post_item = stac_utils.search_single_scene(catalog, BBOX, "2022-01-01/2022-04-30", label="immediate post-fire")
longterm_item = stac_utils.search_single_scene(catalog, BBOX, "2024-06-01/2024-09-15", label="long-term (2024)")

In [ ]:
def load_scene(item):
    ds = odc.stac.load(
        [item], bands=stac_utils.S2_BANDS, bbox=BBOX,
        crs="EPSG:32613", resolution=stac_utils.GSD_M, chunks={"x": 1024, "y": 1024},
    )
    arr_da = ds.to_array(dim="band")
    arr = arr_da.isel(time=0).compute() if "time" in arr_da.dims else arr_da.compute()
    return arr, ds

pre_arr, pre_ds = load_scene(pre_item)
post_arr, post_ds = load_scene(post_item)
longterm_arr, longterm_ds = load_scene(longterm_item)

print("shapes:", pre_arr.shape, post_arr.shape, longterm_arr.shape)
assert pre_arr.shape == post_arr.shape == longterm_arr.shape, "grids didn't align -- check bbox/crs/resolution"

In [ ]:
def true_color(arr):
    rgb = arr.sel(band=["B04", "B03", "B02"]).values.astype("float32")
    return np.clip(rgb / (np.percentile(rgb, 98) + 1e-6), 0, 1).transpose(1, 2, 0)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, arr, item, label in zip(
    axes, [pre_arr, post_arr, longterm_arr], [pre_item, post_item, longterm_item],
    ["pre-fire", "immediate post-fire", "long-term (2024)"],
):
    ax.imshow(true_color(arr))
    ax.set_title(f"{label}\n{item.datetime.date()}")
    ax.axis("off")
plt.tight_layout()
plt.savefig("docs/figures/fire_true_color_dates.png", dpi=150, bbox_inches="tight")
plt.show()
print("Check the middle panel for snow before trusting dNBR below -- a uniformly white/grey")
print("cast over vegetated areas (not just the burn scar) means snow contamination.")

## Chip grid and dNBR

NBR = (NIR &minus; SWIR2) / (NIR + SWIR2); dNBR = NBR_pre &minus; NBR_post. Higher dNBR means a
bigger drop in NBR after the fire, the standard convention for burn severity. A chip is only
kept if it has usable data in **all three** dates, so the pre/post/long-term comparison is on
exactly matched chips throughout.

In [ ]:
NIR, SWIR2 = 6, 9  # indices into stac_utils.S2_BANDS

def nbr(arr):
    nir = arr.values[NIR].astype("float32")
    swir2 = arr.values[SWIR2].astype("float32")
    return (nir - swir2) / (nir + swir2 + 1e-6)

nbr_pre = nbr(pre_arr)
nbr_post = nbr(post_arr)
dnbr = nbr_pre - nbr_post

height, width = pre_arr.shape[1], pre_arr.shape[2]
grid = stac_utils.make_pixel_chip_grid(height, width)
print(f"{len(grid)} candidate chips ({height // stac_utils.CHIP_SIZE_PX} rows x {width // stac_utils.CHIP_SIZE_PX} cols)")

x_coords, y_coords = pre_arr.x.values, pre_arr.y.values
raster_crs = pre_ds.odc.crs
NODATA_FRAC_THRESHOLD = 0.05

chips_meta, pre_pixels, post_pixels, longterm_pixels, mean_dnbr = [], [], [], [], []
for chip in grid:
    ys, xs = chip["y_slice"], chip["x_slice"]
    pre_patch = pre_arr.values[:, ys, xs]
    post_patch = post_arr.values[:, ys, xs]
    lt_patch = longterm_arr.values[:, ys, xs]
    if max(np.isnan(pre_patch).mean(), np.isnan(post_patch).mean(), np.isnan(lt_patch).mean()) > NODATA_FRAC_THRESHOLD:
        continue

    bounds = stac_utils.pixel_window_to_lonlat_bounds(x_coords, y_coords, chip, raster_crs)
    lat, lon = stac_utils.bounds_centroid(bounds)

    chips_meta.append({"id": chip["id"], "bounds": bounds, "lat": lat, "lon": lon})
    pre_pixels.append(np.nan_to_num(pre_patch, nan=0.0).astype("float32"))
    post_pixels.append(np.nan_to_num(post_patch, nan=0.0).astype("float32"))
    longterm_pixels.append(np.nan_to_num(lt_patch, nan=0.0).astype("float32"))
    mean_dnbr.append(float(np.nanmean(dnbr[ys, xs])))

pre_pixels = np.stack(pre_pixels)
post_pixels = np.stack(post_pixels)
longterm_pixels = np.stack(longterm_pixels)
mean_dnbr = np.array(mean_dnbr)
print(f"Kept {len(chips_meta)}/{len(grid)} chips present in all three dates")

In [ ]:
rows = [int(m["id"][1:4]) for m in chips_meta]
cols = [int(m["id"][5:8]) for m in chips_meta]
n_rows, n_cols = max(rows) + 1, max(cols) + 1

dnbr_grid = np.full((n_rows, n_cols), np.nan)
for r, c, val in zip(rows, cols, mean_dnbr):
    dnbr_grid[r, c] = val

plt.figure(figsize=(8, 6))
im = plt.imshow(dnbr_grid, cmap="RdYlGn_r")
plt.colorbar(im, label="dNBR (higher = more severe burn)")
plt.title("Mean dNBR per chip (pre-fire minus immediate post-fire)")
plt.axis("off")
plt.savefig("docs/figures/fire_dnbr_map.png", dpi=150, bbox_inches="tight")
plt.show()

## Embed all three dates with Clay

In [ ]:
ckpt_path = clay_embed.download_checkpoint()
metadata_path = clay_embed.download_metadata_yaml()
model = clay_embed.load_model(ckpt_path, metadata_path, device=device)
wavelengths, band_means, band_stds = clay_embed.load_band_stats(metadata_path)
print("Clay v1.5 loaded")

In [ ]:
def embed_all(pixels, scene_date, chips_meta, batch_size=32):
    dates = [pd.Timestamp(scene_date)] * len(chips_meta)
    lats = [m["lat"] for m in chips_meta]
    lons = [m["lon"] for m in chips_meta]
    out = []
    for start in range(0, len(chips_meta), batch_size):
        end = min(start + batch_size, len(chips_meta))
        batch_pixels = clay_embed.normalize_chips(pixels[start:end], band_means, band_stds)
        time_feats, latlon_feats = clay_embed.make_time_latlon_tensors(
            dates[start:end], lats[start:end], lons[start:end]
        )
        out.append(clay_embed.encode_batch(model, batch_pixels, time_feats, latlon_feats, wavelengths, device=device))
    return np.concatenate(out, axis=0)

emb_pre = embed_all(pre_pixels, pre_item.datetime, chips_meta)
emb_post = embed_all(post_pixels, post_item.datetime, chips_meta)
emb_longterm = embed_all(longterm_pixels, longterm_item.datetime, chips_meta)
print(f"Embedded {len(chips_meta)} chips x 3 dates, dim={emb_pre.shape[1]}")

## The actual test: does embedding movement track burn severity?

Embedding shift = 1 &minus; cosine similarity between a chip's pre-fire and later embedding
(0 = no change, larger = bigger change). If Clay's embeddings are sensitive to the fire, chips
with higher dNBR (burned more severely) should show a larger shift.

In [ ]:
def cosine_sim_rows(a, b):
    a_n = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_n = b / np.linalg.norm(b, axis=1, keepdims=True)
    return (a_n * b_n).sum(axis=1)

shift_immediate = 1 - cosine_sim_rows(emb_pre, emb_post)
shift_longterm = 1 - cosine_sim_rows(emb_pre, emb_longterm)

r_immediate = stats.pearsonr(mean_dnbr, shift_immediate)
r_longterm = stats.pearsonr(mean_dnbr, shift_longterm)

print(f"dNBR vs. embedding shift (pre -> immediate post-fire): r={r_immediate[0]:.3f}, p={r_immediate[1]:.4f}")
print(f"dNBR vs. embedding shift (pre -> 2024 long-term):      r={r_longterm[0]:.3f}, p={r_longterm[1]:.4f}")
print()
print("If r(immediate) is meaningfully higher than r(longterm), that supports 'the signal faded")
print("by 2024' over 'Clay never saw the burn at all' -- resolving notebook 03's open question.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].scatter(mean_dnbr, shift_immediate, s=18, alpha=0.7, color="#f58231")
axes[0].set_xlabel("mean dNBR (burn severity)")
axes[0].set_ylabel("embedding shift, pre -> immediate post-fire")
axes[0].set_title(f"Immediate (r={r_immediate[0]:.2f})")

axes[1].scatter(mean_dnbr, shift_longterm, s=18, alpha=0.7, color="#5ec8ff")
axes[1].set_xlabel("mean dNBR (burn severity)")
axes[1].set_ylabel("embedding shift, pre -> 2024 long-term")
axes[1].set_title(f"Long-term (r={r_longterm[0]:.2f})")

plt.tight_layout()
plt.savefig("docs/figures/fire_dnbr_vs_shift.png", dpi=150)
plt.show()

In [ ]:
shift_immediate_grid = np.full((n_rows, n_cols), np.nan)
shift_longterm_grid = np.full((n_rows, n_cols), np.nan)
for r, c, si, sl in zip(rows, cols, shift_immediate, shift_longterm):
    shift_immediate_grid[r, c] = si
    shift_longterm_grid[r, c] = sl

vmax = max(np.nanmax(shift_immediate_grid), np.nanmax(shift_longterm_grid))
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
im0 = axes[0].imshow(shift_immediate_grid, cmap="inferno", vmin=0, vmax=vmax)
axes[0].set_title("Embedding shift: pre -> immediate")
axes[0].axis("off")
im1 = axes[1].imshow(shift_longterm_grid, cmap="inferno", vmin=0, vmax=vmax)
axes[1].set_title("Embedding shift: pre -> 2024 long-term")
axes[1].axis("off")
fig.colorbar(im1, ax=axes, shrink=0.7, label="1 - cosine similarity")
plt.savefig("docs/figures/fire_shift_maps.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nDone. Commit the new docs/figures/fire_*.png files back to the repo.")
print("(This notebook doesn't touch docs/data/chips.geojson -- it's a separate, self-contained AOI.)")